# HyPhysML — full pipeline on Google Colab

Runs `hyphysml.py` with all outputs and checkpoints written to Google Drive, so an interrupted session resumes where it stopped.

**Setup (once):** create `MyDrive/HyPhysML/` and upload `hyphysml.py` and `FOV dataset.xlsx` (https://doi.org/10.17632/8r7k4cgkg8.1) into it.

**Runtime:** select a GPU runtime (Runtime > Change runtime type). The GPU accelerates XGBoost only; the other learners run on the CPU, so the number of CPU cores matters most (an A100 runtime provides 12). The nested search (11 models × 200 trials × 10 seeds) took about one day on an A100 runtime.

**If the session disconnects:** rerun cells 1, 2, 3 and 5; finished seeds and finished hyperparameter searches are skipped.

In [ ]:
# 1) Mount Google Drive and check the working folder
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/HyPhysML'
os.makedirs(WORK, exist_ok=True)
for f in ['hyphysml.py', 'FOV dataset.xlsx']:
    print(('OK       ' if os.path.isfile(os.path.join(WORK, f)) else 'MISSING  ') + f)

In [ ]:
# 2) Packages and hardware check
!pip -q install optuna shap xgboost lightgbm statsmodels openpyxl
import platform, sklearn, xgboost, lightgbm, optuna, shap, numpy as np
print('python', platform.python_version(), '| sklearn', sklearn.__version__, '| xgboost', xgboost.__version__,
      '| lightgbm', lightgbm.__version__, '| optuna', optuna.__version__, '| shap', shap.__version__)
print('CPU cores:', os.cpu_count())
try:
    xgboost.XGBRegressor(n_estimators=5, device='cuda', tree_method='hist').fit(np.random.rand(50, 3), np.random.rand(50))
    print('XGBoost CUDA: available')
except Exception as e:
    print('XGBoost CUDA not available -> set USE_GPU = False in cell 3:', e)

In [ ]:
# 3) Settings
N_OPTUNA       = 200      # trials per model and seed (value used in the paper)
SEEDS_THIS_RUN = ''       # '' = all 10 seeds; or e.g. '42,7,13'
USE_GPU        = True     # XGBoost on CUDA

os.environ['FOV_DATA'] = os.path.join(WORK, 'FOV dataset.xlsx')
os.environ['FOV_OUT']  = os.path.join(WORK, 'results')
os.environ['HYPHYSML_N_OPTUNA'] = str(N_OPTUNA)
os.environ['HYPHYSML_GPU'] = '1' if USE_GPU else '0'
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['MPLBACKEND'] = 'Agg'

In [ ]:
# 4) Optional smoke test (2 seeds, 2 trials; results go to a separate folder and are NOT the paper results)
RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    !cd "$WORK" && HYPHYSML_FAST=1 FOV_OUT="$WORK/results_FAST_TEST" HYPHYSML_SEEDS= python -u hyphysml.py 2>&1 | tail -40

In [ ]:
# 5) Full run: seeds are split over parallel processes according to the number of CPU cores,
#    then a final pass runs the extrapolation tests and writes all tables and figures.
import subprocess, time, glob, sys
SEEDS_ALL = [42, 7, 13, 99, 2024, 17, 88, 55, 101, 314]
OUT = os.environ['FOV_OUT']; CK = os.path.join(OUT, 'checkpoints'); os.makedirs(CK, exist_ok=True)
todo = [s for s in (map(int, SEEDS_THIS_RUN.split(',')) if SEEDS_THIS_RUN else SEEDS_ALL)
        if not os.path.isfile(os.path.join(CK, f'seed{s}.pkl'))]
cores = os.cpu_count() or 2
n_proc = max(1, min(len(todo), cores // 5)) if todo else 0
cv_jobs = max(1, min(5, cores // max(n_proc, 1)))
groups = [todo[i::n_proc] for i in range(n_proc)]
print(f'{cores} cores -> {n_proc} processes, {cv_jobs} CV jobs each; seed groups: {groups}')
procs = []
for k, g in enumerate(groups):
    env = dict(os.environ, HYPHYSML_SEEDS=','.join(map(str, g)), HYPHYSML_CV_JOBS=str(cv_jobs),
               HYPHYSML_EXTRAPOLATION='0', HYPHYSML_LC_DIAGNOSTIC='0')
    log = open(os.path.join(OUT, f'run_log_proc{k}.txt'), 'a')
    procs.append(subprocess.Popen([sys.executable, '-u', 'hyphysml.py'], cwd=WORK, env=env,
                                  stdout=log, stderr=subprocess.STDOUT))
t0 = time.time()
while procs and any(p.poll() is None for p in procs):
    time.sleep(300)
    done = sorted(int(os.path.basename(x)[4:-4]) for x in glob.glob(os.path.join(CK, 'seed*.pkl')))
    print(f'{(time.time()-t0)/3600:.1f} h | finished seeds: {done}', flush=True)
for k, p in enumerate(procs):
    print(f'process {k} exit code: {p.returncode}' + ('' if p.returncode == 0 else f' -> see run_log_proc{k}.txt'))
missing = [s for s in SEEDS_ALL if not os.path.isfile(os.path.join(CK, f'seed{s}.pkl'))]
if missing:
    print('Missing seeds:', missing, '-> rerun this cell')
else:
    !cd "$WORK" && HYPHYSML_SEEDS= python -u hyphysml.py 2>&1 | tee -a "$WORK/results/run_log_final.txt" | grep -v "^    audit"

In [ ]:
# 6) Progress: finished seeds and finished hyperparameter searches
import json
print('Finished seeds:', sorted(int(os.path.basename(p)[4:-4]) for p in glob.glob(os.path.join(CK, 'seed*.pkl'))))
for p in sorted(glob.glob(os.path.join(CK, 'hpo_seed*.json'))):
    d = json.load(open(p)); done = [k for k in d if not k.startswith('_')]
    print(os.path.basename(p), f'{len(done)}/11:', ', '.join(f"{k} ({d[k]['minutes']} min)" for k in done))

In [ ]:
# 7) Zip and download the results
import shutil
z = shutil.make_archive(os.path.join(WORK, 'results'), 'zip', os.path.join(WORK, 'results'))
from google.colab import files
files.download(z)